In [ ]:
# Sequence features

from nupack import *
import RNA
import math
import itertools

def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

file_paths = [
    'Feature_CNN1.txt'
]

for file_path in file_paths:
    with open(file_path, 'w') as file:
        pass  

# Define NUPACK model
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

# Load crRNA sequences
file1 = open('wang_nc_full_guide_sequences_blank_removed_sampled_1.txt', 'r')    
lines = file1.readlines()
guide_array = [line.strip() for line in lines]

# Load target sequences
file2 = open('wang_nc_target_sequences_noPAM_blank_removed_sampled_1.txt', 'r')    
lines = file2.readlines()
full_target_array = [line.strip() for line in lines]
file3 = open('wang_nc_target_sequences_noPAM_blank_removed_sampled_1.txt', 'r')    
lines = file3.readlines()
truncated_target_array = [line.strip() for line in lines]

for i in range (0, len(guide_array)):

    # Initialize seq_array
    seq_array = []
    seq_array_unit = []

    guide = guide_array[i]
    target_full = full_target_array[i]
    target_truncated = truncated_target_array[i]

    
    for j in range (0, len(guide)):
        if guide[j] == "A":
            to_be_added = [1, 0, 0, 0]
        elif guide[j] == "C":
            to_be_added = [0, 1, 0, 0]
        elif guide[j] == "G":
            to_be_added = [0, 0, 1, 0]
        elif guide[j] == "U":
            to_be_added = [0, 0, 0, 1]

        seq_array_unit.append(to_be_added)

    for k in range (0, len(target_full)):
        if target_full[k] == "A":
            to_be_added = ([1, 0, 0, 0])
        elif target_full[k] == "C":
            to_be_added = ([0, 1, 0, 0])
        elif target_full[k] == "G":
            to_be_added = ([0, 0, 1, 0])
        elif target_full[k] == "T":
            to_be_added = ([0, 0, 0, 1])
        elif target_full[k] == "-":
            to_be_added = ([0, 0, 0, 0])
            
        seq_array_unit.append(to_be_added)

    for k in range (0, len(DNA_reverse_complement(target_full))):
        if DNA_reverse_complement(target_full[k]) == "A":
            to_be_added = ([1, 0, 0, 0])
        elif DNA_reverse_complement(target_full[k]) == "C":
            to_be_added = ([0, 1, 0, 0])
        elif DNA_reverse_complement(target_full[k]) == "G":
            to_be_added = ([0, 0, 1, 0])
        elif DNA_reverse_complement(target_full[k]) == "T":
            to_be_added = ([0, 0, 0, 1])
        elif DNA_reverse_complement(target_full[k]) == "-":
            to_be_added = ([0, 0, 0, 0])
            
        seq_array_unit.append(to_be_added)
            
    seq_array_to_append = [seq_array_unit]
    seq_array.append(seq_array_to_append)

    seq_array = list(itertools.chain.from_iterable(seq_array))
        
    # Open a file in write mode
    with open('Feature_CNN1.txt', 'a') as file:
        # Iterate over each row in the 2D array
        for row in seq_array:
            # Convert each element to a string and join them with spaces
            file.write(' '.join(map(str, row)) + '\n')
        file.write('\n')

print('-------------')

In [ ]:
# Structure features

from nupack import *
import RNA
import math
import itertools

def DNA_to_RNA(DNA):
    match = {'A': 'A', 'C': 'C', 'G': 'G', 'T': 'U'}
    return ''.join(match.get(base, base) for base in (DNA))
            
def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

def get_segments_interrupted_by_plus(input_string: str) -> list:
    # Split the string by the "+" character
    segments = input_string.split("+")
    
    # Remove empty segments if any (e.g., if the string starts or ends with "+")
    segments = [segment for segment in segments if segment]
    
    return segments

file_paths = [
    'Feature_CNN2.txt'
]

for file_path in file_paths:
    with open(file_path, 'w') as file:
        pass  

# Define NUPACK model-same as CRISPR trans-cleavage reaction
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

# Load crRNA sequences
file1 = open('wang_nc_full_guide_sequences_blank_removed_sampled_1.txt', 'r')    
lines = file1.readlines()
guide_array = [line.strip() for line in lines]

# Load target sequences
file2 = open('wang_nc_target_sequences_noPAM_blank_removed_sampled_1.txt', 'r')    
lines = file2.readlines()
full_target_array = [line.strip() for line in lines]
file3 = open('wang_nc_target_sequences_noPAM_blank_removed_sampled_1.txt', 'r')    
lines = file3.readlines()
truncated_target_array = [line.strip() for line in lines]

for i in range (0, len(guide_array)):

    # Initialize struct_array
    struct_array = []
    struct_array_unit = []

    guide = guide_array[i]
    target_full = full_target_array[i]
    target_truncated = truncated_target_array[i]

    # Compute suboptimal structures and energy
    subopt_structures_guide_bh = subopt(strands=guide, energy_gap=3, model=my_model_RNA)  
    subopt_structures_target_bh = subopt(strands=[target_truncated, DNA_reverse_complement(target_truncated)], energy_gap=3, model=my_model_DNA)
    subopt_structures_hybrid_ah = subopt(strands=[guide, target_truncated], energy_gap=3, model=my_model_DNA)
    subopt_structures_target_ah = subopt(strands=DNA_reverse_complement(target_truncated), energy_gap=3, model=my_model_DNA)

    # Guide structure before hybridization
    for j in range (0, 100):
        
        try:
            if str(subopt_structures_guide_bh[0].structure)[j] == '.':
                to_be_added = ([1, 0, 0])
            elif str(subopt_structures_guide_bh[0].structure)[j] == '(':
                to_be_added = ([0, 1, 0])
            elif str(subopt_structures_guide_bh[0].structure)[j] == ')':
                to_be_added = ([0, 0, 1])
            else:
                to_be_added = ([0, 0, 0])
                
        except IndexError:
            to_be_added = ([0, 0, 0])

        struct_array_unit.append(to_be_added)
        
    # Target structure before hybridization 
    target_bh_1 = get_segments_interrupted_by_plus(str(subopt_structures_target_bh[0].structure))[0]
    target_bh_2 = get_segments_interrupted_by_plus(str(subopt_structures_target_bh[0].structure))[1]

    for j in range (0, 20):
        
        try:
            if target_bh_1[j] == '.':
                to_be_added = ([1, 0, 0])
            elif target_bh_1[j] == '(':
                to_be_added = ([0, 1, 0])
            elif target_bh_1[j] == ')':
                to_be_added = ([0, 0, 1])
            else:
                to_be_added = ([0, 0, 0])
            
        except IndexError:
             to_be_added = ([0, 0, 0])
        
        struct_array_unit.append(to_be_added)

        
    for j in range (0, 20):
        
        try:
            if target_bh_2[j] == '.':
                to_be_added = ([1, 0, 0])
            elif target_bh_2[j] == '(':
                to_be_added = ([0, 1, 0])
            elif target_bh_2[j] == ')':
                to_be_added = ([0, 0, 1])
            else:
                to_be_added = ([0, 0, 0])
            
        except IndexError:
             to_be_added = ([0, 0, 0])
        
        struct_array_unit.append(to_be_added)

         
    # Hybrid structure after hybridization 
    hybrid_ah_1 = get_segments_interrupted_by_plus(str(subopt_structures_hybrid_ah[0].structure))[0]
    hybrid_ah_2 = get_segments_interrupted_by_plus(str(subopt_structures_hybrid_ah[0].structure))[1]

    for j in range (0, 100):
        
        try:
            if hybrid_ah_1[j] == '.':
                to_be_added = ([1, 0, 0])
            elif hybrid_ah_1[j] == '(':
                to_be_added = ([0, 1, 0])
            elif hybrid_ah_1[j] == ')':
                to_be_added = ([0, 0, 1])
            else:
                to_be_added = ([0, 0, 0])
            
        except IndexError:
             to_be_added = ([0, 0, 0])
        
        struct_array_unit.append(to_be_added)

     
    for j in range (0, 20):
        
        try:
            if hybrid_ah_2[j] == '.':
                to_be_added = ([1, 0, 0])
                print('yes')
            elif hybrid_ah_2[j] == '(':
                to_be_added = ([0, 1, 0])
            elif hybrid_ah_2[j] == ')':
                to_be_added = ([0, 0, 1])
            else:
                to_be_added = ([0, 0, 0])
            
        except IndexError:
             to_be_added = ([0, 0, 0])
        
        struct_array_unit.append(to_be_added)

    # Target structure after hybridization
    for j in range (0, 20):
        
        try:
            if str(subopt_structures_target_ah[0].structure)[j] == '.':
                to_be_added = ([1, 0, 0])
            elif str(subopt_structures_target_ah[0].structure)[j] == '(':
                to_be_added = ([0, 1, 0])
            elif str(subopt_structures_target_ah[0].structure)[j] == ')':
                to_be_added = ([0, 0, 1])
            else:
                to_be_added = ([0, 0, 0])
                
        except IndexError:
            to_be_added = ([0, 0, 0])

        struct_array_unit.append(to_be_added)

            
    struct_array_to_append = [struct_array_unit]
    struct_array.append(struct_array_to_append)

    struct_array = list(itertools.chain.from_iterable(struct_array))
    
    # Open a file in write mode
    with open('Feature_CNN2.txt', 'a') as file:
        # Iterate over each row in the 2D array
        for row in struct_array:
            # Convert each element to a string and join them with spaces
            file.write(' '.join(map(str, row)) + '\n')
        file.write('\n')


print('-------------')

In [ ]:
# Energy features

from nupack import *
import RNA
import math
import itertools
import numpy as np

def DNA_to_RNA(DNA):
    match = {'A': 'A', 'C': 'C', 'G': 'G', 'T': 'U'}
    return ''.join(match.get(base, base) for base in (DNA))

def RNA_to_DNA(RNA):
    match = {'A': 'A', 'C': 'C', 'G': 'G', 'U': 'T'}
    return ''.join(match.get(base, base) for base in (RNA))
            
def DNA_reverse_complement(DNA):
    complement = {'A': 'T', 'C': 'G', 'G': 'C', 'T': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(DNA))

def RNA_reverse_complement(RNA):
    complement = {'A': 'U', 'C': 'G', 'G': 'C', 'U': 'A'}
    return ''.join(complement.get(base, base) for base in reversed(RNA))

def normalize_guide(value):
    normalized_value = (value - (-20)) / (0 - (-20))
    return normalized_value

def normalize_target(value):
    normalized_value = (value - (-45)) / (-25 - (-45))
    return normalized_value

def normalize_guide_conse(value):
    normalized_value = (value - (-47)) / (3 - (-47))
    return normalized_value

def normalize_target_conse(value):
    normalized_value = (value - (-36)) / (-17 - (-36))
    return normalized_value

def normalize_guide_conse_unpaired(value):
    normalized_value = (value - (-42)) / (3 - (-42))
    return normalized_value

def normalize_target_conse_unpaired(value):
    normalized_value = (value - (-5)) / (0 - (-5))
    return normalized_value

def normalize_guide_overhang(value):
    normalized_value = (value - (-42)) / (3 - (-42))
    return normalized_value

def normalize_target_overhang(value):
    normalized_value = (value - (-5)) / (0 - (-5))
    return normalized_value

def normalize_guide_paired(value):
    normalized_value = (value - (-47)) / (3 - (-47))
    return normalized_value

def normalize_target_paired(value):
    normalized_value = (value - (-36)) / (-17 - (-36))
    return normalized_value

def normalize_seed(value):
    normalized_value = (value - (-11)) / (-3 - (-11))
    return normalized_value

def normalize_middle(value):
    normalized_value = (value - (-14)) / (-4 - (-14))
    return normalized_value

def normalize_distal(value):
    normalized_value = (value - (-13)) / (-4 - (-13))
    return normalized_value

def find_max_base_pairs(string):
    max_length = 0
    current_length = 0
    max_start_index = -1
    current_start_index = -1
    current_char = ''

    for i, char in enumerate(string):
        if char in '()':
            if char == current_char:
                current_length += 1
            else:
                current_char = char
                current_length = 1
                current_start_index = i
        else:
            current_length = 0

        if current_length > max_length:
            max_length = current_length
            max_start_index = current_start_index

    return max_start_index, max_length

def find_max_unpaired(string):
    max_length = 0
    current_length = 0
    max_start_index = -1
    current_start_index = -1
    current_char = ''

    for i, char in enumerate(string):
        if char in '.':
            if char == current_char:
                current_length += 1
            else:
                current_char = char
                current_length = 1
                current_start_index = i
        else:
            current_length = 0

        if current_length > max_length:
            max_length = current_length
            max_start_index = current_start_index

    return max_start_index, max_length

def detect_5_overhang(input_string):

    current_length = 0
    start_index = None
    for i, char in enumerate(input_string):
        if char == '.':
            if current_length == 0:
                start_index = i
            current_length += 1
        else:
            if current_length > 0:  # Found a sequence of dots (either single or multiple)
                return (start_index, current_length)
            current_length = 0

    # Check if the loop ended with a sequence of dots
    if current_length > 0:
        return (start_index, current_length)

    return (None, 0)

def detect_3_overhang(input_string):

    current_length = 0
    start_index = None
    last_start_index = None
    last_length = 0

    for i, char in enumerate(input_string):
        if char == '.':
            if current_length == 0:
                start_index = i
            current_length += 1
        else:
            if current_length > 0:  # Found a sequence of dots
                last_start_index = start_index
                last_length = current_length
                current_length = 0

    # Check if the loop ended with a sequence of dots
    if current_length > 0:
        last_start_index = start_index
        last_length = current_length

    if last_start_index is not None:
        return (last_start_index, last_length)

    return (None, 0)

def detect_5_paired(input_string):

    current_length = 0
    start_index = None
    for i, char in enumerate(input_string):
        if char == '(' or char == ')':
            if current_length == 0:
                start_index = i
            current_length += 1
        else:
            if current_length > 0:  # Found a sequence of dots (either single or multiple)
                return (start_index, current_length)
            current_length = 0

    # Check if the loop ended with a sequence of dots
    if current_length > 0:
        return (start_index, current_length)

    return (None, 0)

def detect_3_paired(input_string):

    current_length = 0
    start_index = None
    last_start_index = None
    last_length = 0

    for i, char in enumerate(input_string):
        if char == '(' or char == ')':
            if current_length == 0:
                start_index = i
            current_length += 1
        else:
            if current_length > 0:  # Found a sequence of dots
                last_start_index = start_index
                last_length = current_length
                current_length = 0

    # Check if the loop ended with a sequence of dots
    if current_length > 0:
        last_start_index = start_index
        last_length = current_length

    if last_start_index is not None:
        return (last_start_index, last_length)

    return (None, 0)

def is_all_parens(s):
    for char in s:
        if char not in ("(", ")"):
            return False
    return True
    

file_paths = [
    'Feature_MLP.txt'
]

for file_path in file_paths:
    with open(file_path, 'w') as file:
        pass  

# Define NUPACK model-same as CRISPR trans-cleavage reaction
my_model_RNA = Model(material='rna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)
my_model_DNA = Model(material='dna', ensemble='nostacking', celsius=37, sodium=0.1, magnesium=0.01)

# Load crRNA sequences
file1 = open('wang_nc_full_guide_sequences_blank_removed_sampled_1.txt', 'r')    
lines = file1.readlines()
guide_array = [line.strip() for line in lines]

# Load target sequences
file2 = open('wang_nc_target_sequences_noPAM_blank_removed_sampled_1.txt', 'r')    
lines = file2.readlines()
full_target_array = [line.strip() for line in lines]
file3 = open('wang_nc_target_sequences_noPAM_blank_removed_sampled_1.txt', 'r')    
lines = file3.readlines()
truncated_target_array = [line.strip() for line in lines]

min_guide_energy = np.inf
max_guide_energy = -np.inf
min_target_energy = np.inf
max_target_energy = -np.inf

min_guide_energy_conse = np.inf
max_guide_energy_conse = -np.inf
min_target_energy_conse = np.inf
max_target_energy_conse = -np.inf

min_guide_energy_conse_unpaired = np.inf
max_guide_energy_conse_unpaired = -np.inf
min_target_energy_conse_unpaired = np.inf
max_target_energy_conse_unpaired = -np.inf

min_guide_energy_5_overhang = np.inf
max_guide_energy_5_overhang = -np.inf
min_target_energy_5_overhang = np.inf
max_target_energy_5_overhang = -np.inf

min_guide_energy_3_overhang = np.inf
max_guide_energy_3_overhang = -np.inf
min_target_energy_3_overhang = np.inf
max_target_energy_3_overhang = -np.inf

min_guide_energy_5_paired = np.inf
max_guide_energy_5_paired = -np.inf
min_target_energy_5_paired = np.inf
max_target_energy_5_paired = -np.inf

min_guide_energy_3_paired = np.inf
max_guide_energy_3_paired = -np.inf
min_target_energy_3_paired = np.inf
max_target_energy_3_paired = -np.inf

max_seed_energy = -np.inf
min_seed_energy = np.inf

max_middle_energy = -np.inf
min_middle_energy = np.inf

max_distal_energy = -np.inf
min_distal_energy = np.inf

 # Initialize energy_array
energy_array = []

for i in range (0, len(guide_array)):
    energy_array_unit = []

    guide = guide_array[i]
    target_full = full_target_array[i]
    target_truncated = truncated_target_array[i]

    # Initialize energy_array
    energy_array_guide = []
    energy_array_target = []

    # Compute ensemble energy
    partition_function_guide = pfunc(strands=guide, model=my_model_RNA)
    ensemble_energy_guide = partition_function_guide[1]
    partition_function_target = pfunc(strands=[guide, target_truncated], model=my_model_DNA)
    ensemble_energy_target = partition_function_target[1]
    
    # Compute suboptimal structures and energy
    subopt_structures_guide = subopt(strands=guide, energy_gap=3, model=my_model_RNA)  
    subopt_structures_target = subopt(strands=[guide, target_truncated], energy_gap=3, model=my_model_DNA)

    len_scaffold = 80
    len_spacer = 20
    
    # Calculate/compare subopt_structures_guide[0].energy and subopt_structures_target[0].energy
    if subopt_structures_guide[0].energy + 13.23 > max_guide_energy:
        max_guide_energy = subopt_structures_guide[0].energy + 13.23
    if subopt_structures_guide[0].energy + 13.23 < min_guide_energy:
        min_guide_energy = subopt_structures_guide[0].energy + 13.23
    if subopt_structures_target[0].energy > max_target_energy:
        max_target_energy = subopt_structures_target[0].energy
    if subopt_structures_target[0].energy < min_target_energy:
        min_target_energy = subopt_structures_target[0].energy
        
    # Calculate/compare ensemble_energy_max_paired_guide and ensemble_energy_max_paired_target
    if str(subopt_structures_guide[0].structure)[0:len_spacer] == '.' * len_spacer:
        ensemble_energy_max_paired_guide = 0
    else:
        max_start_index_guide, max_length_guide = find_max_base_pairs(str(subopt_structures_guide[0].structure)[0:len_spacer])
        max_paired_seq_guide = guide[max_start_index_guide:max_start_index_guide+max_length_guide]
        partition_function_max_paired_guide = pfunc(strands=[max_paired_seq_guide, RNA_reverse_complement(max_paired_seq_guide)], model=my_model_RNA)
        ensemble_energy_max_paired_guide = partition_function_max_paired_guide[1]

    if str(subopt_structures_target[0].structure)[0:len_spacer] == '.' * len_spacer:
        ensemble_energy_max_paired_target = 0
    else:
        max_start_index_target, max_length_target = find_max_base_pairs(str(subopt_structures_target[0].structure)[0:len_spacer])
        max_paired_seq_target = guide[max_start_index_target:max_start_index_target+max_length_target]
        DNA_max_paired_seq_target = RNA_to_DNA(max_paired_seq_target)
        partition_function_max_paired_target = pfunc(strands=[DNA_max_paired_seq_target, DNA_reverse_complement(DNA_max_paired_seq_target)], model=my_model_DNA)
        ensemble_energy_max_paired_target = partition_function_max_paired_target[1]

    if ensemble_energy_max_paired_guide > max_guide_energy_conse:
        max_guide_energy_conse = ensemble_energy_max_paired_guide
    if ensemble_energy_max_paired_guide < min_guide_energy_conse:
        min_guide_energy_conse = ensemble_energy_max_paired_guide

    if ensemble_energy_max_paired_target > max_target_energy_conse:
        max_target_energy_conse = ensemble_energy_max_paired_target
    if ensemble_energy_max_paired_target < min_target_energy_conse:
        min_target_energy_conse = ensemble_energy_max_paired_target

    
    # Calculate/compare ensemble_energy_max_unpaired_guide and ensemble_energy_max_unpaired_target
    if is_all_parens(str(subopt_structures_guide[0].structure)[0:len_spacer]) == True:
        ensemble_energy_max_unpaired_guide = 0
    else:
        max_start_index_guide, max_length_guide = find_max_unpaired(str(subopt_structures_guide[0].structure)[0:len_spacer])
        max_unpaired_seq_guide = guide[max_start_index_guide:max_start_index_guide+max_length_guide]
        partition_function_max_unpaired_guide = pfunc(strands=[max_unpaired_seq_guide, RNA_reverse_complement(max_unpaired_seq_guide)], model=my_model_RNA)
        ensemble_energy_max_unpaired_guide = partition_function_max_unpaired_guide[1]

    if is_all_parens(str(subopt_structures_target[0].structure)[0:len_spacer]) == True:
        ensemble_energy_max_unpaired_target = 0
    else:
        max_start_index_target, max_length_target = find_max_unpaired(str(subopt_structures_target[0].structure)[0:len_spacer])
        max_unpaired_seq_target = guide[max_start_index_target:max_start_index_target+max_length_target]
        DNA_max_unpaired_seq_target = RNA_to_DNA(max_unpaired_seq_target)
        partition_function_max_unpaired_target = pfunc(strands=[DNA_max_unpaired_seq_target, DNA_reverse_complement(DNA_max_unpaired_seq_target)], model=my_model_DNA)
        ensemble_energy_max_unpaired_target = partition_function_max_unpaired_target[1]

    if ensemble_energy_max_unpaired_guide > max_guide_energy_conse_unpaired:
        max_guide_energy_conse_unpaired = ensemble_energy_max_unpaired_guide
    if ensemble_energy_max_unpaired_guide < min_guide_energy_conse_unpaired:
        min_guide_energy_conse_unpaired = ensemble_energy_max_unpaired_guide

    if ensemble_energy_max_unpaired_target > max_target_energy_conse_unpaired:
        max_target_energy_conse_unpaired = ensemble_energy_max_unpaired_target
    if ensemble_energy_max_unpaired_target < min_target_energy_conse_unpaired:
        min_target_energy_conse_unpaired = ensemble_energy_max_unpaired_target

    # Calculate/compare ensemble_energy_5_overhang_guide and ensemble_energy_5_overhang_target
    if str(subopt_structures_guide[0].structure)[0] != '.':
        ensemble_energy_5_overhang_guide = 0
    else:
        max_start_index_guide, max_length_guide = detect_5_overhang(str(subopt_structures_guide[0].structure)[0:len_spacer])
        max_5_overhang_seq_guide = guide[max_start_index_guide:max_start_index_guide+max_length_guide]
        partition_function_5_overhang_guide = pfunc(strands=[max_5_overhang_seq_guide, RNA_reverse_complement(max_5_overhang_seq_guide)], model=my_model_RNA)
        ensemble_energy_5_overhang_guide = partition_function_5_overhang_guide[1]

    if str(subopt_structures_target[0].structure)[0] != '.':
        ensemble_energy_5_overhang_target = 0
    else:
        max_start_index_target, max_length_target = detect_5_overhang(str(subopt_structures_target[0].structure)[0:len_spacer])
        max_5_overhang_seq_target = guide[max_start_index_target:max_start_index_target+max_length_target]
        DNA_5_overhang_seq_target = RNA_to_DNA(max_5_overhang_seq_target)
        partition_function_5_overhang_target = pfunc(strands=[DNA_5_overhang_seq_target, DNA_reverse_complement(DNA_5_overhang_seq_target)], model=my_model_DNA)
        ensemble_energy_5_overhang_target = partition_function_5_overhang_target[1]

    if ensemble_energy_5_overhang_guide > max_guide_energy_5_overhang:
        max_guide_energy_5_overhang = ensemble_energy_5_overhang_guide
    if ensemble_energy_5_overhang_guide < min_guide_energy_5_overhang:
        min_guide_energy_5_overhang = ensemble_energy_5_overhang_guide

    if ensemble_energy_5_overhang_target > max_target_energy_5_overhang:
        max_target_energy_5_overhang = ensemble_energy_5_overhang_target
    if ensemble_energy_5_overhang_target < min_target_energy_5_overhang:
        min_target_energy_5_overhang = ensemble_energy_5_overhang_target
            

    # Calculate/compare ensemble_energy_3_overhang_guide and ensemble_energy_3_overhang_target
    if str(subopt_structures_guide[0].structure)[len_spacer-1] != '.':
        ensemble_energy_3_overhang_guide = 0
    else:
        max_start_index_guide, max_length_guide = detect_3_overhang(str(subopt_structures_guide[0].structure)[0:len_spacer])
        max_3_overhang_seq_guide = guide[max_start_index_guide:max_start_index_guide+max_length_guide]
        partition_function_3_overhang_guide = pfunc(strands=[max_3_overhang_seq_guide, RNA_reverse_complement(max_3_overhang_seq_guide)], model=my_model_RNA)
        ensemble_energy_3_overhang_guide = partition_function_3_overhang_guide[1]

    if str(subopt_structures_target[0].structure)[len_spacer-1] != '.':
        ensemble_energy_3_overhang_target = 0
    else:
        max_start_index_target, max_length_target = detect_3_overhang(str(subopt_structures_target[0].structure)[0:len_spacer])
        max_3_overhang_seq_target = guide[max_start_index_target:max_start_index_target+max_length_target]
        DNA_3_overhang_seq_target = RNA_to_DNA(max_3_overhang_seq_target)
        partition_function_3_overhang_target = pfunc(strands=[DNA_3_overhang_seq_target, DNA_reverse_complement(DNA_3_overhang_seq_target)], model=my_model_DNA)
        ensemble_energy_3_overhang_target = partition_function_3_overhang_target[1]

    if ensemble_energy_3_overhang_guide > max_guide_energy_3_overhang:
        max_guide_energy_3_overhang = ensemble_energy_3_overhang_guide
    if ensemble_energy_3_overhang_guide < min_guide_energy_3_overhang:
        min_guide_energy_3_overhang = ensemble_energy_3_overhang_guide

    if ensemble_energy_3_overhang_target > max_target_energy_3_overhang:
        max_target_energy_3_overhang = ensemble_energy_3_overhang_target
    if ensemble_energy_3_overhang_target < min_target_energy_3_overhang:
        min_target_energy_3_overhang = ensemble_energy_3_overhang_target
            

    # Calculate/compare ensemble_energy_5_paired_guide and ensemble_energy_5_paired_target
    if str(subopt_structures_guide[0].structure)[0] == '.':
        ensemble_energy_5_paired_guide = 0
    else:
        max_start_index_guide, max_length_guide = detect_5_paired(str(subopt_structures_guide[0].structure)[0:len_spacer])
        max_5_paired_seq_guide = guide[max_start_index_guide:max_start_index_guide+max_length_guide]
        partition_function_5_paired_guide = pfunc(strands=[max_5_paired_seq_guide, RNA_reverse_complement(max_5_paired_seq_guide)], model=my_model_RNA)
        ensemble_energy_5_paired_guide = partition_function_5_paired_guide[1]

    if str(subopt_structures_target[0].structure)[0] == '.':
        ensemble_energy_5_paired_target = 0
    else:
        max_start_index_target, max_length_target = detect_5_paired(str(subopt_structures_target[0].structure)[0:len_spacer])
        max_5_paired_seq_target = guide[max_start_index_target:max_start_index_target+max_length_target]
        DNA_5_paired_seq_target = RNA_to_DNA(max_5_paired_seq_target)
        partition_function_5_paired_target = pfunc(strands=[DNA_5_paired_seq_target, DNA_reverse_complement(DNA_5_paired_seq_target)], model=my_model_DNA)
        ensemble_energy_5_paired_target = partition_function_5_paired_target[1]

    if ensemble_energy_5_paired_guide > max_guide_energy_5_paired:
        max_guide_energy_5_paired = ensemble_energy_5_paired_guide
    if ensemble_energy_5_paired_guide < min_guide_energy_5_paired:
        min_guide_energy_5_paired = ensemble_energy_5_paired_guide

    if ensemble_energy_5_paired_target > max_target_energy_5_paired:
        max_target_energy_5_paired = ensemble_energy_5_paired_target
    if ensemble_energy_5_paired_target < min_target_energy_5_paired:
        min_target_energy_5_paired = ensemble_energy_5_paired_target

    # Calculate/compare ensemble_energy_3_paired_guide and ensemble_energy_3_paired_target
    if str(subopt_structures_guide[0].structure)[len_spacer-1] == '.':
        ensemble_energy_3_paired_guide = 0
    else:
        max_start_index_guide, max_length_guide = detect_3_paired(str(subopt_structures_guide[0].structure)[0:len_spacer])
        max_3_paired_seq_guide = guide[max_start_index_guide:max_start_index_guide+max_length_guide]
        partition_function_3_paired_guide = pfunc(strands=[max_3_paired_seq_guide, RNA_reverse_complement(max_3_paired_seq_guide)], model=my_model_RNA)
        ensemble_energy_3_paired_guide = partition_function_3_paired_guide[1]

    if str(subopt_structures_target[0].structure)[len_spacer-1] == '.':
        ensemble_energy_3_paired_target = 0
    else:
        max_start_index_target, max_length_target = detect_3_paired(str(subopt_structures_target[0].structure)[0:len_spacer])
        max_3_paired_seq_target = guide[max_start_index_target:max_start_index_target+max_length_target]
        DNA_3_paired_seq_target = RNA_to_DNA(max_3_paired_seq_target)
        partition_function_3_paired_target = pfunc(strands=[DNA_3_paired_seq_target, DNA_reverse_complement(DNA_3_paired_seq_target)], model=my_model_DNA)
        ensemble_energy_3_paired_target = partition_function_3_paired_target[1]

    if ensemble_energy_3_paired_guide > max_guide_energy_3_paired:
        max_guide_energy_3_paired = ensemble_energy_3_paired_guide
    if ensemble_energy_3_paired_guide < min_guide_energy_3_paired:
        min_guide_energy_3_paired = ensemble_energy_3_paired_guide

    if ensemble_energy_3_paired_target > max_target_energy_3_paired:
        max_target_energy_3_paired = ensemble_energy_3_paired_target
    if ensemble_energy_3_paired_target < min_target_energy_3_paired:
        min_target_energy_3_paired = ensemble_energy_3_paired_target

    # Calculate/compare target seed region free energy 
    seed_region = target_truncated[0:6]
    subopt_structures_seed = subopt(strands=[seed_region, DNA_reverse_complement(seed_region)], energy_gap=3, model=my_model_DNA)
    seed_energy = subopt_structures_seed[0].energy

    if seed_energy > max_seed_energy:
        max_seed_energy = seed_energy
    if seed_energy < min_seed_energy:
        min_seed_energy = seed_energy

    # Calculate/compare target middle region free energy 
    middle_region = target_truncated[6:13]
    subopt_structures_middle = subopt(strands=[middle_region, DNA_reverse_complement(middle_region)], energy_gap=3, model=my_model_DNA)
    middle_energy = subopt_structures_middle[0].energy

    if middle_energy > max_middle_energy:
        max_middle_energy = middle_energy
    if middle_energy < min_middle_energy:
        min_middle_energy = middle_energy

    # Calculate/compare target distal region free energy 
    distal_region = target_truncated[13:20]
    subopt_structures_distal = subopt(strands=[distal_region, DNA_reverse_complement(distal_region)], energy_gap=3, model=my_model_DNA)
    distal_energy = subopt_structures_distal[0].energy

    if distal_energy > max_distal_energy:
        max_distal_energy = distal_energy
    if distal_energy < min_distal_energy:
        min_distal_energy = distal_energy

    to_be_added = [normalize_guide(subopt_structures_guide[0].energy + 13.23)]
    to_be_added.append(normalize_guide_conse(ensemble_energy_max_paired_guide))
    to_be_added.append(normalize_guide_conse_unpaired(ensemble_energy_max_unpaired_guide))
    to_be_added.append(normalize_guide_overhang(ensemble_energy_5_overhang_guide))
    to_be_added.append(normalize_guide_overhang(ensemble_energy_3_overhang_guide))
    to_be_added.append(normalize_guide_paired(ensemble_energy_5_paired_guide))
    to_be_added.append(normalize_guide_paired(ensemble_energy_3_paired_guide))
    
    to_be_added.append(normalize_target(subopt_structures_target[0].energy))
    to_be_added.append(normalize_target_conse(ensemble_energy_max_paired_target))
    to_be_added.append(normalize_target_conse_unpaired(ensemble_energy_max_unpaired_target))
    to_be_added.append(normalize_target_overhang(ensemble_energy_5_overhang_target))
    to_be_added.append(normalize_target_overhang(ensemble_energy_3_overhang_target))
    to_be_added.append(normalize_target_paired(ensemble_energy_5_paired_target))
    to_be_added.append(normalize_target_paired(ensemble_energy_3_paired_target))
    
    to_be_added.append(normalize_seed(seed_energy))
    to_be_added.append(normalize_middle(middle_energy))
    to_be_added.append(normalize_distal(distal_energy))

    energy_array_unit.append(to_be_added)

    energy_array_to_append = [energy_array_unit]
    energy_array.append(energy_array_to_append)

energy_array = list(itertools.chain.from_iterable(energy_array))

# Open a file in write mode
with open('Feature_MLP.txt', 'a') as file:
    # Iterate over each row in the 2D array
    for row in energy_array:
        # Convert each element to a string and join them with spaces
        file.write(' '.join(map(str, row)) + '\n')

        

print('-------------')

print(min_guide_energy, max_guide_energy, min_target_energy, max_target_energy)
print(min_guide_energy_conse, max_guide_energy_conse, min_target_energy_conse, max_target_energy_conse)
print(min_guide_energy_conse_unpaired, max_guide_energy_conse_unpaired, min_target_energy_conse_unpaired, max_target_energy_conse_unpaired)
print(min_guide_energy_5_overhang, max_guide_energy_5_overhang, min_target_energy_5_overhang, max_target_energy_5_overhang)
print(min_guide_energy_3_overhang, max_guide_energy_3_overhang, min_target_energy_3_overhang, max_target_energy_3_overhang)
print(min_guide_energy_5_paired, max_guide_energy_5_paired, min_target_energy_5_paired, max_target_energy_5_paired)
print(min_guide_energy_3_paired, max_guide_energy_3_paired, min_target_energy_3_paired, max_target_energy_3_paired)

print(max_seed_energy, min_seed_energy)
print(max_middle_energy, min_middle_energy)
print(max_distal_energy, min_distal_energy)